# Lab 16-04 · The FFT in Software and Silicon

Five parts. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import shutil
import subprocess
import time
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import fft

## Part 1 · Check and count

In [2]:
rng = np.random.default_rng(0)
print(" N     ok   butterflies  non-trivial  formula")
for N in (2, 4, 8, 16, 64, 256, 1024, 4096):
    x = rng.normal(size=N) + 1j * rng.normal(size=N)
    ok = np.allclose(fft.fft_dit(x), np.fft.fft(x)) and np.allclose(fft.fft_dif(x), np.fft.fft(x))
    b, c, nt = fft.twiddle_counts(N)
    m = int(math.log2(N))
    print(f"{N:5d} {ok!s:5s} {b:11d} {nt:12d} {max(N // 2 * m - 3 * N // 2 + 2, 0):8d}")

 N     ok   butterflies  non-trivial  formula
    2 True            1            0        0
    4 True            4            0        0
    8 True           12            2        2
   16 True           32           10       10
   64 True          192           98       98
  256 True         1024          642      642
 1024 True         5120         3586     3586
 4096 True        24576        18434    18434


## Part 2 · Timing

*Predict:* the slopes of direct DFT, Python FFT and numpy FFT on log-log axes.

In [3]:
Ns = [64, 128, 256, 512, 1024, 2048, 4096]
t_dir, t_py, t_np = [], [], []
for N in Ns:
    x = rng.normal(size=N) + 1j * rng.normal(size=N)
    F = np.exp(-2j * np.pi * np.outer(np.arange(N), np.arange(N)) / N)
    t0 = time.perf_counter(); F @ x; t1 = time.perf_counter()
    fft.fft_dit(x); t2 = time.perf_counter()
    for _ in range(100):
        np.fft.fft(x)
    t3 = time.perf_counter()
    t_dir.append(t1 - t0); t_py.append(t2 - t1); t_np.append((t3 - t2) / 100)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(Ns, t_dir, "o-", label="direct (matrix product, excl. building F)")
ax.loglog(Ns, t_py, "s-", label="fft_dit (Python loops)")
ax.loglog(Ns, t_np, "^-", label="numpy.fft (compiled)")
ax.set_xlabel("N"); ax.set_ylabel("seconds"); ax.legend(fontsize=7); plt.show()
print("slopes (log-log, 1024 to 4096):",
      [round(math.log(t[-1] / t[-3]) / math.log(4), 2) for t in (t_dir, t_py, t_np)])

slopes (log-log, 1024 to 4096): [1.98, 1.21, 0.91]


## Part 3 · Real data and the inverse

In [4]:
a, b = rng.normal(size=256), rng.normal(size=256)
A, B = fft.two_real_ffts(a, b)
print("two real FFTs from one complex FFT:", np.allclose(A, np.fft.fft(a)), np.allclose(B, np.fft.fft(b)))
X = np.fft.fft(a)
print("inverse via forward FFT:", np.allclose(fft.ifft_via_fft(X), a))

two real FFTs from one complex FFT: True True
inverse via forward FFT: True


## Part 4 · The hardware FFT

In [5]:
if shutil.which("iverilog"):
    subprocess.run(["iverilog", "-g2012", "-o", "_fft8.vvp", "fft8_tb.v", "fft8.v", "bfly.v"], check=True)
    print(subprocess.run(["vvp", "-n", "_fft8.vvp"], capture_output=True, text=True).stdout.strip())
    import os; os.remove("_fft8.vvp")
else:
    print("iverilog not installed: skip (CI runs fft8_tb.v through tools/run_hdl.py)")

words = open("vectors.hex").read().split()
s16 = lambda h: int(h, 16) - 65536 if int(h, 16) >= 32768 else int(h, 16)
for t in range(len(words) // 32):
    w = [s16(h) for h in words[32 * t:32 * t + 32]]
    before = fft.SAT_EVENTS[0]
    Xr, Xi = fft.fft8_fixed(w[:8], w[8:16])
    if fft.SAT_EVENTS[0] > before:
        exact = np.fft.fft(np.array(w[:8]) + 1j * np.array(w[8:16])) / 8
        print(f"vector {t} saturates; output vs exact DFT/8:")
        for k in range(8):
            print(f"  bin {k}: ({Xr[k]:6d}, {Xi[k]:6d})   exact ({exact[k].real:9.1f}, {exact[k].imag:9.1f})")

TEST PASSED (66 vectors, bit-exact)
fft8_tb.v:31: $finish called at 66000 (1ps)
vector 5 saturates; output vs exact DFT/8:
  bin 0: (    -1,  -8192)   exact (     -0.5,   -8192.4)
  bin 1: ( -5793,  22176)   exact (  -5792.5,   22176.3)
  bin 2: (  8191,      0)   exact (   8191.9,       0.0)
  bin 3: (   993,  -5793)   exact (    993.8,   -5792.5)
  bin 4: (     0,   8191)   exact (      0.0,    8191.9)
  bin 5: (  5792,  10591)   exact (   5792.5,   10591.2)
  bin 6: ( -8192,     -1)   exact (  -8191.9,       0.0)
  bin 7: (-32768,   5792)   exact ( -33761.3,    5792.5)


## Part 5 · Fixed-point error

In [6]:
rng_lab = np.random.default_rng(1640)
errs = []
for _ in range(2000):
    xr = list(map(int, rng_lab.integers(-16000, 16001, 8)))
    xi = list(map(int, rng_lab.integers(-16000, 16001, 8)))
    Xr, Xi = fft.fft8_fixed(xr, xi)
    errs.append(np.max(np.abs(np.array(Xr) + 1j * np.array(Xi) - np.fft.fft(np.array(xr) + 1j * np.array(xi)) / 8)))
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(errs, bins=40); ax.set_xlabel("largest error per vector (LSB)"); plt.show()
print(f"max {max(errs):.2f} LSB, mean {np.mean(errs):.2f} LSB")

max 2.03 LSB, mean 1.29 LSB


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.